In [0]:
path="/Volumes/main/default/selic/selic.parquet"

In [0]:
from pyspark.sql.types import DoubleType
from pyspark.sql import functions as F
df_selic=spark.read.parquet(path)


df_selic = df_selic.withColumn("data", F.to_date("data", "dd/MM/yyyy"))
df_selic = df_selic.withColumn("valor", F.col("valor").cast(DoubleType()))
df_selic.display()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F



df_selic = df_selic.withColumn("ano",       F.year("data"))
df_selic = df_selic.withColumn("trimestre", F.quarter("data"))
df_selic = df_selic.withColumn("mes",       F.month("data"))


mes       = Window.partitionBy("ano", "mes")
trimestre = Window.partitionBy("ano", "trimestre")
ano       = Window.partitionBy("ano")
d30       = Window.partitionBy("ano").orderBy("data").rowsBetween(-29, 0)
d90       = Window.partitionBy("ano").orderBy("data").rowsBetween(-89, 0)
lag       = Window.partitionBy("ano").orderBy("data")


df_selic = df_selic.withColumn("media_mensal",    F.round(F.avg("valor").over(mes), 4))
df_selic = df_selic.withColumn("media_trimestre", F.round(F.avg("valor").over(trimestre), 4))
df_selic = df_selic.withColumn("media_anual",     F.round(F.avg("valor").over(ano), 4))

df_selic = df_selic.withColumn("media_movel_30d", F.round(F.avg("valor").over(d30), 4))
df_selic = df_selic.withColumn("media_movel_90d", F.round(F.avg("valor").over(d90), 4))

df_selic = df_selic.withColumn("valor_anterior",  F.round(F.lag("valor", 1).over(lag), 4))
df_selic = df_selic.withColumn("variacao_diaria", F.round(F.col("valor") - F.col("valor_anterior"), 4))
df_selic = df_selic.withColumn("variacao_pct",    F.round((F.col("variacao_diaria") / F.col("valor_anterior")) * 100, 4))

df_selic.display()

In [0]:
df_selic = df_selic.dropna()

df_selic.display()


In [0]:
# CARREGANDO EM UMA TABELA DELTA ANTES DE EXPORTAR PARA O POWER BI
df_selic.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("main.default.selic_gold")